# Лабораторная работа 5. Нейросетевая классификация отзывов Кинопоиска

В этой лабораторной работе решается задача классификации тональности отзывов Кинопоиска с помощью нейронных сетей на PyTorch.

## Содержание

1. Окружение и настройки
2. Загрузка датасета
3. Разведочный анализ данных
4. Предобработка текста
5. Токенизация, padding и cache encoded tensors
6. Нейросетевые модели
7. Обучение, calibration и сравнение качества
8. Анализ ошибок
9. Проверка на новых текстах
10. Выводы

## 1. Окружение и настройки

Сначала подключаем библиотеки и фиксируем все параметры эксперимента. Устройство выбирается автоматически: `cuda`, затем `mps`, затем `cpu`.

### 1.1 Импорты

Подключаем только те библиотеки, которые используются дальше: работа с данными, визуализация, PyTorch, spaCy и метрики качества.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import warnings
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spacy
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

### 1.2 Константы эксперимента

Здесь собраны все настройки, которые обычно хочется менять между запусками: режим предобработки, размеры последовательностей, batch size и параметры обучения.

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
px.defaults.template = "plotly_white"

SEED = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_RU = {
    "negative": "негативный",
    "neutral": "нейтральный",
    "positive": "позитивный",
}

TEST_SIZE = 0.20
VALID_SIZE = 0.15
PREPROCESS_MODE = "soft"  # варианты: "soft" или "lemma"

MAX_VOCAB_SIZE = 60_000
MIN_FREQ = 2
CNN_MAX_LEN = 512
GRU_MAX_LEN = 400
CNN_BATCH_SIZE = 512
GRU_BATCH_SIZE = 256
EVAL_BATCH_MULTIPLIER = 2

MAX_EPOCHS = 10
PATIENCE = 3
CNN_LR = 1e-3
GRU_LR = 7e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.03

EMBEDDING_DIM = 200
CNN_NUM_FILTERS = 128
CNN_KERNEL_SIZES = (2, 3, 4, 5, 7)
GRU_HIDDEN_SIZE = 192
DROPOUT = 0.4

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

PREPROCESSING_VERSION = "lab5_full_train_pytorch_v2"
SPACY_MODEL = "ru_core_news_sm"
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"

### 1.3 Воспроизводимость и устройство

Фиксируем seed и печатаем итоговое устройство. AMP включается только на CUDA, чтобы не ломать запуск на CPU или Apple MPS.

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(SEED)
DEVICE = resolve_device()
AMP_ENABLED = DEVICE.type == "cuda"
DATALOADER_NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"AMP enabled: {AMP_ENABLED}")
print(f"DataLoader workers: {DATALOADER_NUM_WORKERS}")

## 2. Загрузка датасета

Используется тот же подготовленный датасет отзывов. Если файла нет в `LAB_V/data/dataset`, он создается hardlink-ом из `LAB_IV`, а если hardlink недоступен — обычным копированием.

### 2.1 Пути к данным

Находим корень проекта и проверяем, что рабочий CSV лежит в папке пятой лабораторной.

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "LAB_V").exists() and (candidate / "LAB_IV").exists():
            return candidate
    raise RuntimeError("Не удалось найти корень проекта с папками LAB_IV и LAB_V")


PROJECT_ROOT = find_project_root(Path.cwd())
LAB_IV_DIR = PROJECT_ROOT / "LAB_IV"
LAB_V_DIR = PROJECT_ROOT / "LAB_V"
DATASET_DIR = LAB_V_DIR / "data" / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME
SOURCE_DATA_PATH = LAB_IV_DIR / "data" / "dataset" / PREPARED_DATA_FILENAME

if not DATA_PATH.exists():
    if not SOURCE_DATA_PATH.exists():
        raise FileNotFoundError(f"Не найден исходный датасет: {SOURCE_DATA_PATH}")
    try:
        os.link(SOURCE_DATA_PATH, DATA_PATH)
        link_mode = "hardlink"
    except OSError:
        shutil.copy2(SOURCE_DATA_PATH, DATA_PATH)
        link_mode = "copy"
    print(f"Created dataset {link_mode}: {DATA_PATH}")
else:
    print(f"Dataset exists: {DATA_PATH}")

### 2.2 Чтение таблицы

Считываем подготовленный CSV, оставляем нужные классы и добавляем стабильный `row_uid` для проверки кэшей.

In [ ]:
df = pd.read_csv(DATA_PATH)
required_columns = {"review", "sentiment"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"В датасете не хватает колонок: {sorted(missing_columns)}")

df = df.dropna(subset=["review", "sentiment"]).copy()
df = df[df["sentiment"].isin(LABEL_ORDER)].reset_index(drop=True)

uid_columns = [column for column in ["source_file", "movie_id", "review_id"] if column in df.columns]
df["row_uid"] = df[uid_columns].astype(str).agg("|".join, axis=1) if uid_columns else df.index.astype(str)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df["sentiment"].value_counts().reindex(LABEL_ORDER).to_frame("count"))

## 3. Разведочный анализ данных

EDA выполняется по полному датасету. Это важно: train будет полным, но перед обучением полезно понять естественное распределение классов и длин отзывов.

### 3.1 Размеры отзывов

Считаем длины в символах и словах, чтобы дальше осознанно выбрать `MAX_LEN` и head-tail truncation.

In [ ]:
eda_df = df.copy()
eda_df["char_len"] = eda_df["review"].astype(str).str.len()
eda_df["word_count"] = eda_df["review"].astype(str).str.split().map(len)

display(
    eda_df.groupby("sentiment")[["char_len", "word_count"]]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

### 3.2 Распределение классов

Проверяем исходный дисбаланс. В новой схеме он сохраняется в train, validation и test, а компенсируется через loss.

In [ ]:
class_counts = eda_df["sentiment"].value_counts().reindex(LABEL_ORDER).reset_index()
class_counts.columns = ["sentiment", "count"]
class_counts["label"] = class_counts["sentiment"].map(LABEL_RU)

fig = px.bar(
    class_counts,
    x="label",
    y="count",
    text="count",
    title="Распределение классов в полном датасете",
    labels={"label": "Класс", "count": "Количество отзывов"},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

### 3.3 Распределение длин

Показываем хвосты распределения длин. Для рецензий важно не брать только начало текста, потому что итоговая оценка часто появляется в конце.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Символы", "Слова"))
fig.add_trace(go.Histogram(x=eda_df["char_len"], nbinsx=80, name="Символы"), row=1, col=1)
fig.add_trace(go.Histogram(x=eda_df["word_count"], nbinsx=80, name="Слова"), row=1, col=2)
fig.update_layout(title="Распределение длин отзывов", showlegend=False)
fig.show()

percentiles = [50, 75, 90, 95, 99]
length_stats = pd.DataFrame(
    {
        "percentile": percentiles,
        "words": np.percentile(eda_df["word_count"], percentiles).round(0).astype(int),
        "chars": np.percentile(eda_df["char_len"], percentiles).round(0).astype(int),
    }
)
display(length_stats)

### 3.4 Частотные слова

Быстро смотрим частотные слова по классам. Этот блок нужен не для модели, а для проверки тональных маркеров в корпусе.

In [ ]:
EDA_STOPWORDS = set(SPACY_RU_STOPWORDS) | {"это", "все", "ещё", "еще", "который", "фильм"}
class_word_counters: dict[str, Counter] = {}

for label in LABEL_ORDER:
    counter = Counter()
    for text in eda_df.loc[eda_df["sentiment"] == label, "review"].astype(str):
        tokens = re.findall(r"[а-яА-ЯёЁa-zA-Z]+", text.lower())
        counter.update(token for token in tokens if token not in EDA_STOPWORDS and len(token) >= 3)
    class_word_counters[label] = counter

top_words_df = pd.DataFrame(
    [
        {"sentiment": label, "word": word, "count": count}
        for label, counter in class_word_counters.items()
        for word, count in counter.most_common(15)
    ]
)

fig = px.bar(
    top_words_df,
    x="count",
    y="word",
    color="sentiment",
    facet_col="sentiment",
    orientation="h",
    title="Топ слов по классам",
    height=520,
)
fig.update_yaxes(matches=None, categoryorder="total ascending")
fig.show()

## 4. Предобработка текста

Основной режим — мягкая очистка `soft`. Лемматизация через spaCy оставлена как переключаемый baseline, но по умолчанию нейросети получают менее агрессивно обработанный текст.

### 4.1 Стоп-слова и важные тональные маркеры

Удаляем обычные стоп-слова, но сохраняем слова, которые меняют тональность: отрицания, усилители и явные оценки.

In [ ]:
IMPORTANT_WORDS = {
    "не", "нет", "ни", "но",
    "очень", "слишком", "совсем",
    "плохо", "хорошо", "ужасно", "прекрасно",
    "скучно", "интересно",
    "лучше", "хуже",
}

STOPWORDS = set(SPACY_RU_STOPWORDS) - IMPORTANT_WORDS

print(f"Preprocessing mode: {PREPROCESS_MODE}")
print(f"Stopwords: {len(STOPWORDS)}")
print({word: (word in STOPWORDS) for word in sorted(IMPORTANT_WORDS)})

### 4.2 Ленивая загрузка spaCy

Модель spaCy загружается только при выборе `PREPROCESS_MODE = "lemma"`, поэтому обычный `soft`-запуск не тратит на это время.

In [ ]:
nlp = None


def get_nlp():
    global nlp
    if nlp is None:
        try:
            nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
        except OSError as exc:
            raise RuntimeError(
                "Не найдена модель ru_core_news_sm. Установите её командой: "
                "uv run python -m spacy download ru_core_news_sm"
            ) from exc
    return nlp

### 4.3 Функции очистки текста

Все варианты предобработки проходят через один интерфейс. Эти же функции потом используются для новых пользовательских отзывов.

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[^а-яa-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def preprocess_text_soft(text: str) -> str:
    return " ".join(
        token
        for token in normalize_text(text).split()
        if len(token) >= 2 and token not in STOPWORDS
    )


def clean_doc_lemma(doc) -> str:
    lemmas = []
    for token in doc:
        lemma = token.lemma_.lower().strip().replace("ё", "е")
        if lemma and len(lemma) >= 2 and lemma not in STOPWORDS and not token.is_space:
            lemmas.append(lemma)
    return " ".join(lemmas)


def preprocess_text(text: str) -> str:
    if PREPROCESS_MODE == "soft":
        return preprocess_text_soft(text)
    if PREPROCESS_MODE == "lemma":
        return clean_doc_lemma(get_nlp()(normalize_text(text)))
    raise ValueError(f"Неизвестный PREPROCESS_MODE: {PREPROCESS_MODE}")


def preprocess_texts(texts: pd.Series | list[str], batch_size: int = 512, n_process: int = 1) -> list[str]:
    text_list = list(texts)
    if PREPROCESS_MODE == "soft":
        return [preprocess_text_soft(text) for text in tqdm(text_list, desc="soft preprocessing")]
    if PREPROCESS_MODE == "lemma":
        normalized = [normalize_text(text) for text in text_list]
        docs = get_nlp().pipe(normalized, batch_size=batch_size, n_process=n_process)
        return [clean_doc_lemma(doc) for doc in tqdm(docs, total=len(normalized), desc="spaCy preprocessing")]
    raise ValueError(f"Неизвестный PREPROCESS_MODE: {PREPROCESS_MODE}")

### 4.4 Быстрая проверка предобработки

Смотрим один пример до и после очистки, чтобы сразу заметить слишком агрессивные правила.

In [ ]:
example_text = df.loc[0, "review"]
print("До:", example_text[:350])
print("После:", preprocess_text(example_text)[:350])

### 4.5 Train/validation/test split

Делим данные стратифицированно и сохраняем естественное распределение классов. Балансировка удалением объектов здесь не используется.

In [ ]:
raw_train_df, test_raw_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["sentiment"],
)
train_raw_df, val_raw_df = train_test_split(
    raw_train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=raw_train_df["sentiment"],
)

raw_splits = {
    "train": train_raw_df.reset_index(drop=True),
    "validation": val_raw_df.reset_index(drop=True),
    "test": test_raw_df.reset_index(drop=True),
}
model_raw_df = pd.concat(
    [frame.assign(split=split_name) for split_name, frame in raw_splits.items()],
    ignore_index=True,
)

split_summary = pd.DataFrame(
    {
        split_name: frame["sentiment"].value_counts().reindex(LABEL_ORDER)
        for split_name, frame in raw_splits.items()
    }
)
split_summary.loc["total"] = split_summary.sum(axis=0)

display(split_summary.astype(int))
print(f"Model rows: {len(model_raw_df):,}")

### 4.6 Метаданные clean-cache

Кэш предобработки привязан к режиму очистки, версии pipeline и `row_uid` внутри каждого split.

In [ ]:
CLEAN_CACHE_PATH = DATASET_DIR / f"kinopoisk_reviews_cleaned_{PREPROCESS_MODE}.parquet"
CLEAN_CACHE_META_PATH = DATASET_DIR / f"kinopoisk_reviews_cleaned_{PREPROCESS_MODE}.meta.json"


def row_uid_hash(row_uids: pd.Series) -> str:
    payload = "\n".join(row_uids.astype(str)).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def expected_clean_cache_meta(raw_frame: pd.DataFrame) -> dict:
    return {
        "version": PREPROCESSING_VERSION,
        "preprocess_mode": PREPROCESS_MODE,
        "rows": int(len(raw_frame)),
        "row_uid_hash": row_uid_hash(raw_frame["row_uid"]),
        "split_uid_hashes": {
            split_name: row_uid_hash(raw_frame.loc[raw_frame["split"] == split_name, "row_uid"])
            for split_name in ["train", "validation", "test"]
        },
        "important_words": sorted(IMPORTANT_WORDS),
        "spacy_model": SPACY_MODEL if PREPROCESS_MODE == "lemma" else None,
    }


def clean_cache_is_valid(raw_frame: pd.DataFrame) -> bool:
    if not CLEAN_CACHE_PATH.exists() or not CLEAN_CACHE_META_PATH.exists():
        return False
    try:
        actual = json.loads(CLEAN_CACHE_META_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    return actual == expected_clean_cache_meta(raw_frame)

### 4.7 Запуск или чтение предобработки

Если валидный кэш уже есть, используем его. Иначе очищаем тексты и сохраняем результат в parquet.

In [ ]:
if clean_cache_is_valid(model_raw_df):
    model_df = pd.read_parquet(CLEAN_CACHE_PATH)
    print(f"Loaded preprocessing cache: {CLEAN_CACHE_PATH}")
else:
    model_df = model_raw_df.copy()
    model_df["cleaned_text"] = preprocess_texts(model_df["review"], batch_size=512, n_process=1)
    model_df.to_parquet(CLEAN_CACHE_PATH, index=False)
    CLEAN_CACHE_META_PATH.write_text(
        json.dumps(expected_clean_cache_meta(model_raw_df), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved preprocessing cache: {CLEAN_CACHE_PATH}")

model_df["cleaned_word_count"] = model_df["cleaned_text"].str.split().map(len)
model_df = model_df[model_df["cleaned_word_count"] > 0].reset_index(drop=True)

display(model_df[["review", "cleaned_text", "sentiment", "split"]].head())
display(model_df.groupby(["split", "sentiment"])["cleaned_word_count"].agg(["count", "mean", "median"]).round(2))

## 5. Токенизация, padding и cache encoded tensors

Словарь строится только по train-части. Для каждой длины последовательности сохраняется отдельный cache encoded tensors.

### 5.1 Финальные split-таблицы

После удаления пустых очищенных текстов заново выделяем train, validation и test.

In [ ]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"] == "validation"].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)
split_frames = {"train": train_df, "validation": val_df, "test": test_df}

length_stats = pd.DataFrame(
    {
        "split": list(split_frames),
        "rows": [len(frame) for frame in split_frames.values()],
        "p95_cleaned_words": [int(np.percentile(frame["cleaned_word_count"], 95)) for frame in split_frames.values()],
        "max_len_for_cnn": CNN_MAX_LEN,
        "max_len_for_gru": GRU_MAX_LEN,
    }
)
display(length_stats)

### 5.2 Vocabulary по train

Собираем словарь через `Counter`: `<PAD>` получает индекс 0, `<UNK>` — индекс 1, остальные токены идут по частоте.

In [ ]:
def iter_tokens(texts: pd.Series):
    for text in texts:
        yield from str(text).split()


def build_vocab(texts: pd.Series, max_vocab_size: int = MAX_VOCAB_SIZE, min_freq: int = MIN_FREQ):
    counter = Counter(iter_tokens(texts))
    words = [word for word, count in counter.most_common(max_vocab_size - 2) if count >= min_freq]
    word_to_idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    word_to_idx.update({word: idx for idx, word in enumerate(words, start=2)})
    return word_to_idx, counter


def make_vocab_hash(word_to_idx: dict[str, int]) -> str:
    payload = json.dumps(word_to_idx, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


word_to_idx, token_counter = build_vocab(train_df["cleaned_text"])
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
label_to_id = {label: idx for idx, label in enumerate(LABEL_ORDER)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
VOCAB_HASH = make_vocab_hash(word_to_idx)

print(f"Vocabulary size: {len(word_to_idx):,}")
print(f"Vocabulary hash: {VOCAB_HASH}")
print("Most common tokens:", token_counter.most_common(15))
print("Labels:", label_to_id)

### 5.3 Кодирование текстов

Длинные отзывы кодируются через head-tail truncation: берется начало и конец текста, а не только первые токены.

In [ ]:
def truncate_head_tail(ids: list[int], max_len: int) -> list[int]:
    if len(ids) <= max_len:
        return ids
    head_len = max_len // 2
    return ids[:head_len] + ids[-(max_len - head_len):]


def encode_text(text: str, word_to_idx: dict[str, int], max_len: int) -> tuple[list[int], int]:
    token_ids = [word_to_idx.get(token, UNK_IDX) for token in str(text).split()]
    token_ids = truncate_head_tail(token_ids, max_len) or [UNK_IDX]
    length = min(len(token_ids), max_len)
    return token_ids + [PAD_IDX] * (max_len - length), length


def encode_texts(texts: pd.Series | list[str], word_to_idx: dict[str, int], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    sequences, lengths = zip(*(encode_text(text, word_to_idx, max_len) for text in texts))
    return torch.tensor(sequences, dtype=torch.long), torch.tensor(lengths, dtype=torch.long)


def labels_for(frame: pd.DataFrame) -> torch.Tensor:
    return torch.tensor(frame["sentiment"].map(label_to_id).to_numpy(), dtype=torch.long)

### 5.4 Encoded cache

Для CNN и GRU используются разные `MAX_LEN`, поэтому кэш хранится отдельно по длине, режиму предобработки и hash словаря.

In [ ]:
ENCODED_CACHE_DIR = DATASET_DIR / "encoded_cache"
ENCODED_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def expected_encoded_meta(max_len: int) -> dict:
    return {
        "version": PREPROCESSING_VERSION,
        "preprocess_mode": PREPROCESS_MODE,
        "vocab_hash": VOCAB_HASH,
        "max_len": int(max_len),
        "split_uid_hashes": {name: row_uid_hash(frame["row_uid"]) for name, frame in split_frames.items()},
    }


def encoded_cache_path(max_len: int) -> Path:
    return ENCODED_CACHE_DIR / f"encoded_{PREPROCESS_MODE}_{VOCAB_HASH}_len{max_len}.pt"


def load_or_encode_splits(max_len: int) -> dict:
    path = encoded_cache_path(max_len)
    expected_meta = expected_encoded_meta(max_len)
    if path.exists():
        cached = torch.load(path, map_location="cpu")
        if cached.get("meta") == expected_meta:
            print(f"Loaded encoded tensors: {path}")
            return cached

    encoded = {"meta": expected_meta}
    for split_name, frame in split_frames.items():
        sequences, lengths = encode_texts(frame["cleaned_text"], word_to_idx, max_len=max_len)
        encoded[split_name] = {
            "sequences": sequences,
            "lengths": lengths,
            "labels": labels_for(frame),
        }
    torch.save(encoded, path)
    print(f"Saved encoded tensors: {path}")
    return encoded

### 5.5 Подготовка encoded tensors

Загружаем или создаем encoded tensors сразу для двух длин: 512 для CNN и 400 для Packed BiGRU.

In [ ]:
encoded_by_max_len = {
    CNN_MAX_LEN: load_or_encode_splits(CNN_MAX_LEN),
    GRU_MAX_LEN: load_or_encode_splits(GRU_MAX_LEN),
}

### 5.6 Dataset и DataLoader

`Dataset` возвращает последовательность, реальную длину и метку. Длины нужны Packed BiGRU, а CNN их просто игнорирует.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences: torch.Tensor, lengths: torch.Tensor, labels: torch.Tensor):
        self.sequences = sequences.long()
        self.lengths = lengths.long()
        self.labels = labels.long()

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return self.sequences[idx], self.lengths[idx], self.labels[idx]

### 5.7 DataLoader

Для CUDA включаем `pin_memory` и persistent workers, чтобы GPU меньше простаивала во время обучения.

In [ ]:
def make_loader(encoded_split: dict, batch_size: int, shuffle: bool = False) -> DataLoader:
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": DATALOADER_NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
    }
    if DATALOADER_NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = True
    return DataLoader(
        ReviewDataset(encoded_split["sequences"], encoded_split["lengths"], encoded_split["labels"]),
        **loader_kwargs,
    )


def make_loaders_for(max_len: int, train_batch_size: int) -> dict[str, DataLoader]:
    encoded = encoded_by_max_len[max_len]
    return {
        "train": make_loader(encoded["train"], batch_size=train_batch_size, shuffle=True),
        "validation": make_loader(encoded["validation"], batch_size=train_batch_size * EVAL_BATCH_MULTIPLIER),
        "test": make_loader(encoded["test"], batch_size=train_batch_size * EVAL_BATCH_MULTIPLIER),
    }

### 5.8 DataLoader для каждой архитектуры

CNN и GRU используют разные длины и batch size, поэтому loaders собираются отдельно.

In [ ]:
loaders_by_model = {
    "TextCNNMulti": make_loaders_for(CNN_MAX_LEN, CNN_BATCH_SIZE),
    "PackedBiGRU": make_loaders_for(GRU_MAX_LEN, GRU_BATCH_SIZE),
}

for model_name, loaders in loaders_by_model.items():
    print(model_name, {split: len(loader) for split, loader in loaders.items()})

## 6. Нейросетевые модели

Сравниваются две архитектуры: быстрый multi-kernel CNN и recurrent baseline с Packed BiGRU.

### 6.1 Multi-kernel TextCNN

CNN смотрит на n-граммы разной длины через свертки с kernel size `(2, 3, 4, 5, 7)` и объединяет признаки через GlobalMaxPool.

In [ ]:
class TextCNNMulti(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_filters: int = CNN_NUM_FILTERS,
        kernel_sizes: tuple[int, ...] = CNN_KERNEL_SIZES,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList(
            nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size)
            for kernel_size in kernel_sizes
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor | None = None) -> torch.Tensor:
        embedded = self.embedding(x).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            features = F.relu(conv(embedded))
            pooled.append(F.max_pool1d(features, kernel_size=features.size(2)).squeeze(2))
        return self.classifier(self.dropout(torch.cat(pooled, dim=1)))

### 6.2 Packed BiGRU

BiGRU получает реальные длины последовательностей и не читает PAD-токены как часть текста.

In [ ]:
class TextPackedBiGRU(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = EMBEDDING_DIM,
        hidden_size: int = GRU_HIDDEN_SIZE,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.gru(packed)
        features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(features))

### 6.3 Конфигурации моделей

Описываем параметры запуска в одном списке, чтобы обучение и оценка дальше не зависели от конкретного класса модели.

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


model_configs = [
    {
        "name": "TextCNNMulti",
        "factory": lambda: TextCNNMulti(len(word_to_idx), len(LABEL_ORDER)),
        "max_len": CNN_MAX_LEN,
        "batch_size": CNN_BATCH_SIZE,
        "lr": CNN_LR,
    },
    {
        "name": "PackedBiGRU",
        "factory": lambda: TextPackedBiGRU(len(word_to_idx), len(LABEL_ORDER)),
        "max_len": GRU_MAX_LEN,
        "batch_size": GRU_BATCH_SIZE,
        "lr": GRU_LR,
    },
]

for config in model_configs:
    model = config["factory"]()
    print(f"{config['name']}: {count_parameters(model):,} trainable parameters")

## 7. Обучение, calibration и сравнение качества

Обучаем модели на полном train-наборе. Для борьбы с дисбалансом используются `sqrt` class weights и label smoothing.

### 7.1 Метрики и class weights

Основная метрика — `macro-F1`. Веса классов считаются только по train-части и передаются в `CrossEntropyLoss`.

In [ ]:
def labels_from_ids(label_ids: np.ndarray | list[int]) -> list[str]:
    return [id_to_label[int(label_id)] for label_id in label_ids]


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


class_counts = train_df["sentiment"].value_counts().reindex(LABEL_ORDER)
class_weights = np.sqrt(class_counts.sum() / (len(LABEL_ORDER) * class_counts))
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32, device=DEVICE)

display(
    pd.DataFrame(
        {
            "label": LABEL_ORDER,
            "train_count": class_counts.values.astype(int),
            "sqrt_class_weight": class_weights.values,
        }
    )
)


def make_criterion() -> nn.Module:
    return nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)

### 7.2 AMP и перенос batch на устройство

AMP включается только на CUDA. На CPU и MPS используется обычный fp32-режим.

In [ ]:
def autocast_context():
    if not AMP_ENABLED:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type="cuda")
    return torch.cuda.amp.autocast()


def make_grad_scaler():
    if not AMP_ENABLED:
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=True)
        except TypeError:
            return torch.amp.GradScaler(enabled=True)
    return torch.cuda.amp.GradScaler(enabled=True)


def move_batch_to_device(batch):
    sequences, lengths, labels = batch
    non_blocking = DEVICE.type == "cuda"
    return (
        sequences.to(DEVICE, non_blocking=non_blocking),
        lengths.to(DEVICE, non_blocking=non_blocking),
        labels.to(DEVICE, non_blocking=non_blocking),
    )

### 7.3 Optimizer step

Выносим backward, gradient clipping и AMP-step в одну функцию, чтобы сам epoch читался короче.

In [ ]:
def optimizer_step(loss: torch.Tensor, model: nn.Module, optimizer: torch.optim.Optimizer, scaler=None) -> None:
    if scaler is not None and scaler.is_enabled():
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

### 7.4 Один проход обучения или оценки

`run_epoch` используется для train и validation во время early stopping.

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scaler=None,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    y_true_batches = []
    y_pred_batches = []

    for batch in loader:
        sequences, lengths, labels = move_batch_to_device(batch)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train), autocast_context():
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)

        if is_train:
            optimizer_step(loss, model, optimizer, scaler)

        total_loss += loss.item() * labels.size(0)
        y_true_batches.append(labels.detach().cpu().numpy())
        y_pred_batches.append(logits.argmax(dim=1).detach().cpu().numpy())

    y_true = np.concatenate(y_true_batches)
    y_pred = np.concatenate(y_pred_batches)
    return {**compute_metrics(y_true, y_pred), "loss": total_loss / len(loader.dataset)}

### 7.5 Logit bias

Bias применяется только к копии logits: сырые logits сохраняются для дальнейшего анализа и calibration.

In [ ]:
def apply_logit_bias(logits: torch.Tensor, bias: np.ndarray | None = None) -> torch.Tensor:
    adjusted = logits.detach().float()
    if bias is not None:
        adjusted = adjusted + torch.as_tensor(bias, dtype=adjusted.dtype, device=adjusted.device)
    return adjusted

### 7.6 Универсальная оценка модели

`evaluate_model` возвращает метрики, logits, вероятности, предсказания и истинные метки.

In [ ]:
@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module | None = None,
    bias: np.ndarray | None = None,
) -> dict:
    model.eval()
    total_loss = 0.0
    y_true_batches, y_pred_batches, logit_batches, probability_batches = [], [], [], []

    for batch in loader:
        sequences, lengths, labels = move_batch_to_device(batch)
        with autocast_context():
            logits = model(sequences, lengths)
            loss = criterion(logits, labels) if criterion is not None else None

        adjusted_logits = apply_logit_bias(logits, bias)
        probabilities = torch.softmax(adjusted_logits, dim=1)
        predictions = adjusted_logits.argmax(dim=1)

        if loss is not None:
            total_loss += loss.item() * labels.size(0)
        y_true_batches.append(labels.detach().cpu().numpy())
        y_pred_batches.append(predictions.detach().cpu().numpy())
        logit_batches.append(logits.detach().float().cpu().numpy())
        probability_batches.append(probabilities.detach().cpu().numpy())

    y_true = np.concatenate(y_true_batches)
    y_pred = np.concatenate(y_pred_batches)
    metrics = compute_metrics(y_true, y_pred)
    if criterion is not None:
        metrics["loss"] = total_loss / len(loader.dataset)

    return {
        "metrics": metrics,
        "labels": y_true,
        "predictions": y_pred,
        "logits": np.vstack(logit_batches),
        "probabilities": np.vstack(probability_batches),
    }

### 7.7 Обучение с early stopping

Early stopping смотрит строго на validation `macro-F1`, потому что accuracy на несбалансированном корпусе может быть обманчивой.

In [ ]:
def train_model(model: nn.Module, config: dict, loaders: dict[str, DataLoader], criterion: nn.Module) -> dict:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=WEIGHT_DECAY)
    scaler = make_grad_scaler()
    best_state = copy.deepcopy(model.state_dict())
    best_val_macro_f1 = -math.inf
    bad_epochs = 0
    history = []
    started_at = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, loaders["train"], criterion, optimizer=optimizer, scaler=scaler)
        val_metrics = evaluate_model(model, loaders["validation"], criterion=criterion)["metrics"]
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        }
        history.append(row)
        print(
            f"{config['name']} | epoch {epoch:02d} | "
            f"train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f} | "
            f"val_macro_f1={row['val_macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Early stopping after epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_val_macro_f1": best_val_macro_f1,
        "training_time_sec": time.perf_counter() - started_at,
    }

### 7.8 Подбор logit bias

Bias подбирается только на validation logits. Эта корректировка помогает сдвинуть границы классов без повторного обучения модели.

In [ ]:
def find_best_logit_bias(
    val_logits: np.ndarray,
    val_y: np.ndarray,
    grid: np.ndarray | None = None,
) -> tuple[np.ndarray, float]:
    grid = np.linspace(-1.5, 1.5, 31) if grid is None else grid
    best_score = -math.inf
    best_bias = np.zeros(len(LABEL_ORDER), dtype=np.float32)

    for b_negative in grid:
        for b_neutral in grid:
            bias = np.array([b_negative, b_neutral, 0.0], dtype=np.float32)
            score = f1_score(val_y, (val_logits + bias).argmax(axis=1), average="macro", zero_division=0)
            if score > best_score:
                best_score = score
                best_bias = bias
    return best_bias, best_score

### 7.9 Inference для новых текстов

Предсказание для новых отзывов проходит через тот же preprocessing, vocab, padding и calibration bias.

In [ ]:
@torch.inference_mode()
def predict_texts(
    model: nn.Module,
    texts: pd.Series | list[str],
    word_to_idx: dict[str, int],
    max_len: int,
    bias: np.ndarray | None = None,
    batch_size: int = 256,
) -> pd.DataFrame:
    cleaned_texts = preprocess_texts(list(texts), batch_size=512, n_process=1)
    sequences, lengths = encode_texts(cleaned_texts, word_to_idx, max_len=max_len)
    dummy_labels = torch.zeros(len(cleaned_texts), dtype=torch.long)
    loader = make_loader(
        {"sequences": sequences, "lengths": lengths, "labels": dummy_labels},
        batch_size=batch_size,
    )

    model.eval()
    probability_batches = []
    for batch in loader:
        sequences_batch, lengths_batch, _ = move_batch_to_device(batch)
        with autocast_context():
            logits = model(sequences_batch, lengths_batch)
        probabilities = torch.softmax(apply_logit_bias(logits, bias), dim=1).detach().cpu().numpy()
        probability_batches.append(probabilities)

    probabilities = np.vstack(probability_batches)
    result = pd.DataFrame(probabilities, columns=[f"p_{label}" for label in LABEL_ORDER])
    result.insert(0, "predicted_label", labels_from_ids(probabilities.argmax(axis=1)))
    result.insert(0, "cleaned_text", cleaned_texts)
    return result

### 7.10 Обучение одной модели

Функция возвращает саму модель, validation-сводку и историю обучения.

In [ ]:
def train_and_calibrate(config: dict) -> tuple[dict, dict, pd.DataFrame]:
    model_name = config["name"]
    print(f"\nTraining {model_name}")
    loaders = loaders_by_model[model_name]
    model = config["factory"]().to(DEVICE)
    criterion = make_criterion()
    train_result = train_model(model, config, loaders, criterion)
    model = train_result["model"].to(DEVICE)

    val_raw = evaluate_model(model, loaders["validation"], criterion=criterion)
    best_bias, _ = find_best_logit_bias(val_raw["logits"], val_raw["labels"])
    val_calibrated = evaluate_model(model, loaders["validation"], criterion=criterion, bias=best_bias)

    bundle = {
        "model": model,
        "config": config,
        "criterion": criterion,
        "best_bias": best_bias,
        "val_raw": val_raw,
        "val_calibrated": val_calibrated,
        "training_time_sec": train_result["training_time_sec"],
    }
    row = {
        "model": model_name,
        "best_epoch_val_macro_f1": train_result["best_val_macro_f1"],
        "raw_val_macro_f1": val_raw["metrics"]["macro_f1"],
        "calibrated_val_macro_f1": val_calibrated["metrics"]["macro_f1"],
        "best_bias": best_bias.round(3).tolist(),
        "training_time_sec": train_result["training_time_sec"],
        "params": count_parameters(model),
    }
    history = train_result["history"].assign(model=model_name)
    return bundle, row, history

### 7.11 Запуск обучения и calibration

Запускаем обе архитектуры и сохраняем validation-сводку для сравнения.

In [ ]:
trained_models = {}
validation_rows = []
history_frames = []

for config in model_configs:
    bundle, row, history = train_and_calibrate(config)
    trained_models[config["name"]] = bundle
    validation_rows.append(row)
    history_frames.append(history)

validation_summary_df = pd.DataFrame(validation_rows).sort_values("calibrated_val_macro_f1", ascending=False)
history_df = pd.concat(history_frames, ignore_index=True)
display(validation_summary_df)

### 7.12 Динамика validation macro-F1

График нужен, чтобы увидеть, где модель перестала улучшаться и насколько стабильно работал early stopping.

In [ ]:
fig = px.line(
    history_df,
    x="epoch",
    y="val_macro_f1",
    color="model",
    markers=True,
    title="Validation macro-F1 по эпохам",
)
fig.show()

### 7.13 Оценка на test

На test считаем raw и calibrated-качество. Лучшая модель выбирается по calibrated `macro-F1`.

In [ ]:
def collect_test_results(model_name: str, bundle: dict) -> tuple[dict, list[dict]]:
    model = bundle["model"].to(DEVICE)
    loaders = loaders_by_model[model_name]
    raw_result = evaluate_model(model, loaders["test"], criterion=bundle["criterion"])
    calibrated_result = evaluate_model(model, loaders["test"], criterion=bundle["criterion"], bias=bundle["best_bias"])
    rows = []
    for mode_name, result in {"raw": raw_result, "calibrated": calibrated_result}.items():
        rows.append(
            {
                "model": model_name,
                "prediction_mode": mode_name,
                **result["metrics"],
                "training_time_sec": bundle["training_time_sec"],
                "params": count_parameters(model),
                "bias": bundle["best_bias"].round(3).tolist() if mode_name == "calibrated" else [0.0, 0.0, 0.0],
            }
        )
    return {"raw": raw_result, "calibrated": calibrated_result}, rows


test_predictions = {}
test_rows = []
for model_name, bundle in trained_models.items():
    predictions, rows = collect_test_results(model_name, bundle)
    test_predictions[model_name] = predictions
    test_rows.extend(rows)

results_df = pd.DataFrame(test_rows).sort_values(["prediction_mode", "macro_f1"], ascending=[True, False])
calibrated_results_df = results_df[results_df["prediction_mode"] == "calibrated"].sort_values("macro_f1", ascending=False)
best_model_name = calibrated_results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]["model"]
best_bias = trained_models[best_model_name]["best_bias"]
best_result = test_predictions[best_model_name]["calibrated"]
best_config = trained_models[best_model_name]["config"]

display(results_df.reset_index(drop=True))
print(f"Best model by calibrated test macro-F1: {best_model_name}")

### 7.14 Classification report и confusion matrix

Для лучшей calibrated-модели смотрим качество по каждому классу и матрицу ошибок.

In [ ]:
report_dict = classification_report(
    best_result["labels"],
    best_result["predictions"],
    labels=list(range(len(LABEL_ORDER))),
    target_names=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
display(report_df.round(4))

cm = confusion_matrix(best_result["labels"], best_result["predictions"], labels=list(range(len(LABEL_ORDER))))
cm_df = pd.DataFrame(cm, index=[f"true_{label}" for label in LABEL_ORDER], columns=[f"pred_{label}" for label in LABEL_ORDER])
display(cm_df)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Confusion matrix: {best_model_name} calibrated",
)
fig.show()

## 8. Анализ ошибок

Выбираем несколько уверенных ошибок лучшей calibrated-модели и добавляем короткий комментарий, почему промах мог произойти.

### 8.1 Вспомогательные функции анализа

Сокращаем длинные отзывы и формируем простое объяснение ошибки по уверенности и пересечению частотных слов.

In [ ]:
def shorten_text(text: str, max_chars: int = 260) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= max_chars else text[: max_chars - 3] + "..."


def comment_error(row: pd.Series) -> str:
    confidence = row["pred_confidence"]
    tokens = set(str(row["cleaned_text"]).split())
    true_words = {word for word, _ in class_word_counters[row["true_label"]].most_common(80)}
    pred_words = {word for word, _ in class_word_counters[row["pred_label"]].most_common(80)}
    if confidence < 0.45:
        return "Низкая уверенность: вероятно, смешанная или короткая тональность."
    if len(tokens & pred_words) > len(tokens & true_words):
        return "В очищенном тексте больше маркеров предсказанного класса, чем истинного."
    return "Ошибка требует ручного чтения: возможны сарказм, контекст или неоднозначная оценка."

### 8.2 Примеры ошибок

Показываем 3-5 ошибок с максимальной уверенностью модели, потому что именно они лучше всего подсвечивают слабые места pipeline.

In [ ]:
analysis_df = test_df.copy()
analysis_df["true_label"] = labels_from_ids(best_result["labels"])
analysis_df["pred_label"] = labels_from_ids(best_result["predictions"])
analysis_df["pred_confidence"] = best_result["probabilities"].max(axis=1)
analysis_df["review_short"] = analysis_df["review"].map(shorten_text)
analysis_df["cleaned_short"] = analysis_df["cleaned_text"].map(lambda text: shorten_text(text, 240))

errors_df = analysis_df[analysis_df["true_label"] != analysis_df["pred_label"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False).head(5)

if len(errors_df) == 0:
    display(Markdown("Ошибок на test-выборке не найдено."))
else:
    errors_df["comment"] = errors_df.apply(comment_error, axis=1)
    display(
        errors_df[
            ["true_label", "pred_label", "pred_confidence", "review_short", "cleaned_short", "comment"]
        ].style.format({"pred_confidence": "{:.3f}"})
    )

## 9. Проверка на новых текстах

Для демонстрации inference берем пять коротких отзывов в стиле Кинопоиска и прогоняем их через тот же preprocessing, vocab и calibrated-модель.

### 9.1 Новые отзывы

У каждого примера есть ожидаемая метка, чтобы визуально сравнить ее с предсказанием модели.

In [ ]:
new_examples = [
    {
        "review": "Фильм удивительно теплый: актеры играют живо, финал трогает, а история долго не отпускает.",
        "expected_label": "positive",
    },
    {
        "review": "Сценарий разваливается на глазах, диалоги деревянные, а два часа кажутся бесконечными.",
        "expected_label": "negative",
    },
    {
        "review": "Обычная картина на вечер: есть несколько удачных сцен, но ничего принципиально нового она не предлагает.",
        "expected_label": "neutral",
    },
    {
        "review": "Не ожидал такого сильного впечатления: режиссура уверенная, музыка точная, персонажам веришь.",
        "expected_label": "positive",
    },
    {
        "review": "Визуально красиво, но сюжет пустой, герои раздражают, пересматривать точно не хочется.",
        "expected_label": "negative",
    },
]
new_reviews_df = pd.DataFrame(new_examples)

### 9.2 Предсказания на новых отзывах

Используем лучшую calibrated-модель и выводим вероятности по всем трем классам.

In [ ]:
new_predictions = predict_texts(
    best_model.to(DEVICE),
    new_reviews_df["review"],
    word_to_idx,
    max_len=best_config["max_len"],
    bias=best_bias,
    batch_size=len(new_reviews_df),
)
new_result_df = new_reviews_df.join(new_predictions)
display(new_result_df.style.format({column: "{:.3f}" for column in new_result_df.columns if column.startswith("p_")}))

## 10. Выводы

Итоговый вывод формируется после запуска ноутбука, чтобы в тексте были реальные метрики лучшей модели.

### 10.1 Итоговая интерпретация

Вывод фиксирует не только победителя, но и смысл изменений: полный train, class weights, head-tail truncation и calibration.

In [ ]:
best_row = calibrated_results_df.iloc[0]
second_row = calibrated_results_df.iloc[1] if len(calibrated_results_df) > 1 else None

comparison_sentence = ""
if second_row is not None:
    diff = best_row["macro_f1"] - second_row["macro_f1"]
    comparison_sentence = (
        f" Вторая модель, `{second_row['model']}`, отстала по calibrated `macro-F1` на `{diff:.4f}`, "
        "поэтому разница здесь оценивается не по accuracy, а именно по более честной для дисбаланса метрике."
    )

summary_text = f"""
## Итоговые выводы

Лучшей моделью по calibrated `macro-F1` на test-выборке стала `{best_row['model']}`: `macro-F1 = {best_row['macro_f1']:.4f}`, `accuracy = {best_row['accuracy']:.4f}` и `weighted-F1 = {best_row['weighted_f1']:.4f}`.{comparison_sentence}

В этой версии нейросети обучаются на полном train-наборе с естественным распределением классов. Дисбаланс компенсируется мягче: через `sqrt` class weights в `CrossEntropyLoss` и `label_smoothing = {LABEL_SMOOTHING}`. По сути, данные не урезаются перед обучением, а validation и test остаются похожими на реальные данные.

Отдельно полезными изменениями стали head-tail truncation и calibration logits по validation. Для CNN используется `MAX_LEN = {CNN_MAX_LEN}`, для Packed BiGRU — `MAX_LEN = {GRU_MAX_LEN}`, поэтому длинные рецензии не обрываются только по началу текста. Calibration не заменяет нормальное обучение, но помогает поправить границы классов без подглядывания в test. Следующий разумный шаг — взять лучшую архитектуру из этого сравнения и отдельно проверить fastText embeddings.
""".strip()

display(Markdown(summary_text))